# Model Training

In [19]:
import pandas as pd
import numpy as np 

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score, precision_score, recall_score


## Load data

In [20]:
ipi_wide_model = pd.read_csv("data/ipi_wide_full.csv", parse_dates = ["date"], dtype = {"division": str})

## Walk-Forward Validation (6 month gaps)

Three folds:
- Fold 1;
  - Trains on: 2016 to Sep 2022
  - Gap (not used): Oct 2022 to Mar 2023
  - Tested on: Apr 2023 to March 2024

- Fold 2;
  - Trains on: 2016 to Sep 2023
  - Gap (not used): Oct 2023 to Mar 2024
  - Tested on: Apr 2024 to March 2025

- Fold 2;
  - Trains on: 2016 to Sep 2024
  - Gap (not used): Oct 2024 to Mar 2025
  - Tested on: Apr 2025 to March 2026

A six-month gap was imposed between the training and evaluation predictor dates because the target represents distress status six months ahead. This ensures that the outcome months associated with training observations precede the first evaluation month, reducing the risk of look-ahead bias.

In [21]:
features = ["growth_yoy", "yoy_lag_1", "yoy_lag_3", "yoy_roll_6", "mom_vol_6", "distress"]
model_df = ipi_wide_model.dropna(subset = features + ["target"]).copy()
model_df["target"] = model_df["target"].astype(int)

# Test folds (test_start, test_end)
folds = [("2023-04-01", "2024-03-01"), 
          ("2024-04-01", "2025-03-01"),
          ("2025-04-01", "2026-03-01")]


for test_start, test_end in folds:
  # treat values in folds as datetime (pd.Timestamp)
  test_start = pd.Timestamp(test_start) 
  test_end = pd.Timestamp(test_end)

  # Define train and test splits
  train = model_df[model_df["date"] < test_start - pd.DateOffset(months=6)] # train set is less than 6 months before test_start
  test = model_df[(model_df["date"] >= test_start) & (model_df["date"] <= test_end)] 

  X_train, y_train = train[features], train["target"]
  X_test, y_test = test[features], test["target"]

  # test start, training rows, test rows, distress cases in each case
  print(test_start.date(), len(train), len(test), int(y_train.sum()), int(y_test.sum()))


print(ipi_wide_model[features + ["target"]].isna().sum())
print(len(ipi_wide_model), len(model_df))

2023-04-01 1824 288 161 18
2024-04-01 2112 288 208 37
2025-04-01 2400 288 217 20
growth_yoy    288
yoy_lag_1     312
yoy_lag_3     360
yoy_roll_6    408
mom_vol_6     144
distress      336
target        336
dtype: int64
3384 2832


How much of the training positives are COVID?

In [22]:
covid = (train["date"] >= "2020-03-01") & (train["date"] <= "2021-12-01")
print(int(train.loc[covid, "target"].sum()), "of", int(train["target"].sum()))

88 of 217


## Baseline Model

In [23]:
parts = [] # empty list to collect each fold's test rows

for test_start, test_end in folds:
  test_start, test_end = pd.Timestamp(test_start), pd.Timestamp(test_end)

  # Only test since baseline model 
  test = model_df[(model_df["date"] >= test_start) & (model_df["date"] <= test_end)]

  parts.append(test[["date", "division", "distress", "target"]]) # keep 4 columns and add to the list

pooled = pd.concat(parts) # all 3 tables (one from each test fold) into one table

### Evaluation of Baseline Model

In [24]:
print("cases:", int(pooled["target"].sum()))

# score(y_true: ground truth target values, y_pred: estimated targets)
# Since the baseline has no model, "distress" is also used as the answer, hence y_pred
# * Precision: Out of all the predicted positive cases, how many were actually correct?
# * Recall: Out of all the actual positive cases, how many were correctly identified?
# * F1 Score: Harmonic mean of precision and recall

print("precision:", precision_score(pooled["target"], pooled["distress"]))
print("recall:", recall_score(pooled["target"], pooled["distress"]))
print("F1 Score:", f1_score(pooled["target"], pooled["distress"]))

cases: 75
precision: 0.18446601941747573
recall: 0.25333333333333335
F1 Score: 0.21348314606741572


## Decision Tree Classifier (with COVID)

In [25]:
from sklearn import tree

In [26]:
parts_dt = []

for test_start, test_end in folds:
  test_start, test_end = pd.Timestamp(test_start), pd.Timestamp(test_end)

  train = model_df[
    model_df["date"] < test_start - pd.DateOffset(months = 6)
  ] # training data: 6 months before the start of testing data

  test = model_df[
    (model_df["date"] >= test_start) & (model_df["date"] <= test_end)
  ].copy()

  clf = tree.DecisionTreeClassifier(max_depth = 3, random_state = 50)
  clf.fit(train[features], train["target"])

  # for every row in test, use the model to predict the probability of distress six months later
  test["prob"] = clf.predict_proba(test[features])[:, 1]
  
  parts_dt.append(test[["date", "division", "target", "prob"]])

pooled_dt = pd.concat(parts_dt)


### Evaluation of DT Classifier

In [44]:
print("base rate:", pooled_dt["target"].mean())
print("Average Precision Score:   ", average_precision_score(pooled_dt["target"], pooled_dt["prob"]))

# What happens when it actually issues warnings (TP)
for th in [0.5, 0.3, 0.2, 0.12, 0.13, 0.1]: # multiple thresholds
  pred = (pooled_dt["prob"] >= th).astype(int) # warning above thresholds

  print("\n", th, "|", "warnings", pred.sum(), "|",
        "precision:", round(precision_score(pooled_dt["target"], pred), 2), "|",
        "recall:", round(recall_score(pooled_dt["target"], pred), 2))

print("\n")
print("Length of Test:", len(pooled_dt))                 # 864 rows
print("Number of real distress cases:", pooled_dt["target"].sum())      # 75 real distress cases

base rate: 0.08680555555555555
Average Precision Score:    0.18599967477331808

 0.5 | warnings 5 | precision: 0.4 | recall: 0.03

 0.3 | warnings 14 | precision: 0.36 | recall: 0.07

 0.2 | warnings 162 | precision: 0.21 | recall: 0.45

 0.12 | warnings 162 | precision: 0.21 | recall: 0.45

 0.13 | warnings 162 | precision: 0.21 | recall: 0.45

 0.1 | warnings 167 | precision: 0.21 | recall: 0.47


Length of Test: 864
Number of real distress cases: 75


## Decision Tree (without COVID)

In [28]:
parts_dt_x = []

for test_start, test_end in folds:
  test_start, test_end = pd.Timestamp(test_start), pd.Timestamp(test_end)

  train = model_df[
    model_df["date"] < test_start - pd.DateOffset(months = 6)
  ] # training data: 6 months before the start of testing data

  test = model_df[
    (model_df["date"] >= test_start) & (model_df["date"] <= test_end)
  ].copy()

  covid_rows = (train["date"] >= "2019-11-01") & (train["date"] <= "2021-12-01")
  train_clean = train[~covid_rows]

  clf = tree.DecisionTreeClassifier(max_depth = 3, random_state = 50)
  clf.fit(train_clean[features], train_clean["target"])

  # for every row in test, use the model to predict the probability of distress six months later
  test["prob"] = clf.predict_proba(test[features])[:, 1]
  
  parts_dt_x.append(test[["date", "division", "target", "prob"]])

pooled_dt_x = pd.concat(parts_dt_x)  # Define COVID dates (from November 2019 to December 2021)


### Evaluation of DT (without COVID)

In [43]:
print("base rate:", pooled_dt_x["target"].mean())
print("Average Precision Score:   ", average_precision_score(pooled_dt_x["target"], pooled_dt_x["prob"]))

# What happens when it actually issues warnings (TP)
for th in [0.5, 0.3, 0.2, 0.12, 0.13, 0.1]: # multiple thresholds
  pred = (pooled_dt_x["prob"] >= th).astype(int) # warning above thresholds

  print("\n", th, "|", "warnings", pred.sum(), "|",
        "precision:", round(precision_score(pooled_dt_x["target"], pred), 2), "|",
        "recall:", round(recall_score(pooled_dt_x["target"], pred), 2))

print("\n")
print("Length of Test:", len(pooled_dt_x))                 # 864 rows
print("Number of real distress cases:", pooled_dt_x["target"].sum()) 

base rate: 0.08680555555555555
Average Precision Score:    0.1526147960757184

 0.5 | warnings 12 | precision: 0.17 | recall: 0.03

 0.3 | warnings 22 | precision: 0.23 | recall: 0.07

 0.2 | warnings 60 | precision: 0.3 | recall: 0.24

 0.12 | warnings 73 | precision: 0.3 | recall: 0.29

 0.13 | warnings 73 | precision: 0.3 | recall: 0.29

 0.1 | warnings 78 | precision: 0.28 | recall: 0.29


Length of Test: 864
Number of real distress cases: 75


## XGBoost (with COVID)

In [30]:
import xgboost as xgb

In [31]:
parts_xgb = []

for test_start, test_end in folds:
  test_start, test_end = pd.Timestamp(test_start), pd.Timestamp(test_end)

  train = model_df[model_df["date"] < test_start - pd.DateOffset(months=6)]
  test = model_df[(model_df["date"] >= test_start) & (model_df["date"] <= test_end)].copy()

  xgb_model = xgb.XGBClassifier(max_depth = 3, n_estimators = 200, learning_rate = 0.05)

  xgb_model.fit(train[features], train["target"])

  test["prob"] = xgb_model.predict_proba(test[features])[:, 1]

  parts_xgb.append(test[["date", "division", "target", "prob"]])

pooled_xgb = pd.concat(parts_xgb)


# 10 riskiest rows
test.sort_values("prob", ascending=False)[["date", "division", "target", "prob"]].head(10)


,date,division,target,prob
2813,2026-03-01,29,0,0.578470
2803,2025-05-01,29,0,0.453305
1111,2025-05-01,17,1,0.389767
2812,2026-02-01,29,0,0.384468
1825,2026-02-01,22,0,0.257173
1402,2026-02-01,19,0,0.243791
690,2025-07-01,14,0,0.243458
1112,2025-06-01,17,1,0.242238
550,2025-08-01,13,0,0.236903
1824,2026-01-01,22,0,0.232906


### Evaluation of XGBoost (With COVID)

In [ ]:
print("base rate:", pooled_xgb["target"].mean())
print("PR-AUC:   ", average_precision_score(pooled_xgb["target"], pooled_xgb["prob"]))

# What happens when it actually issues warnings (TP)
for th in [0.5, 0.3, 0.2, 0.12, 0.13, 0.1]: # multiple thresholds
  pred = (pooled_xgb["prob"] >= th).astype(int) # warning above thresholds

  print("\n", th, "|", "warnings", pred.sum(), "|",
        "precision:", round(precision_score(pooled_xgb["target"], pred), 2), "|",
        "recall:", round(recall_score(pooled_xgb["target"], pred), 2))

print("\n")
print("Length of Test:", len(pooled_xgb))                 # 864 rows
print("Number of real distress cases:", pooled_xgb["target"].sum())      # 75 real distress cases

base rate: 0.08680555555555555
PR-AUC:    0.18352364064974436

 0.5 | warnings 5 | precision: 0.2 | recall: 0.01

 0.3 | warnings 31 | precision: 0.19 | recall: 0.08

 0.2 | warnings 91 | precision: 0.23 | recall: 0.28

 0.12 | warnings 195 | precision: 0.19 | recall: 0.51

 0.13 | warnings 181 | precision: 0.2 | recall: 0.49

 0.1 | warnings 233 | precision: 0.2 | recall: 0.61


Length of Test: 864
Number of real distress cases: 75


- The model gives each row a probability of distress 6 months later.
- Cutoff turns probabilities into warnings: The lower the cutoff, the more warnings you get, but more false alarms. (tradeoff between precision and recall)
- Increasing Recall reduces False Negatives but may increase False Positives

- At threshold 0.2, the model catches about 28% of real distress cases (Recall; Out of all the actual positive cases, how many were correctly identified). 

- At threshold 0.1 the model catches about 61% of real distress cases.

Decision: Increase Recall even though it increase the amount of False Positives. Precision is almost the same for all rows.

## XGBoost (without COVID)

In [33]:
parts_xgb_x = []

for test_start, test_end in folds:
  test_start, test_end = pd.Timestamp(test_start), pd.Timestamp(test_end)

  train = model_df[model_df["date"] < test_start - pd.DateOffset(months = 6)]
  test = model_df[(model_df["date"] >= test_start) & (model_df["date"] <= test_end)].copy()

  # Define COVID dates (from November 2019 to December 2021)
  covid_rows = (train["date"] >= "2019-11-01") & (train["date"] <= "2021-12-01")
  train_clean = train[~covid_rows]

  xgb_model_x = xgb.XGBClassifier(max_depth = 3, n_estimators = 200, learning_rate = 0.05)

  xgb_model_x.fit(train_clean[features], train_clean["target"])

  test["prob"] = xgb_model_x.predict_proba(test[features])[:, 1]

  parts_xgb_x.append(test[["date", "division", "target", "prob"]])

pooled_xgb_x = pd.concat(parts_xgb_x)


### Evaluation

In [42]:
print("Base Rate:", pooled_xgb_x["target"].mean())
print("PR-AUC:", average_precision_score(pooled_xgb_x["target"], pooled_xgb_x["prob"]))

# What happens when it actually issues warnings (TP)
for th in [0.5, 0.3, 0.2, 0.12, 0.13, 0.1]: # multiple thresholds
  pred = (pooled_xgb_x["prob"] >= th).astype(int) # warning above thresholds

  print("\n", th, "|", "warnings", pred.sum(), "|",
        "precision:", round(precision_score(pooled_xgb_x["target"], pred), 2), "|",
        "recall:", round(recall_score(pooled_xgb_x["target"], pred), 2))

print("\n")
print("Length of Test:", len(pooled_xgb_x))                 # 864 rows
print("Number of real distress cases:", pooled_xgb_x["target"].sum())      # 75 real distress cases

Base Rate: 0.08680555555555555
PR-AUC: 0.17622904258608507

 0.5 | warnings 9 | precision: 0.22 | recall: 0.03

 0.3 | warnings 27 | precision: 0.26 | recall: 0.09

 0.2 | warnings 50 | precision: 0.2 | recall: 0.13

 0.12 | warnings 79 | precision: 0.27 | recall: 0.28

 0.13 | warnings 75 | precision: 0.27 | recall: 0.27

 0.1 | warnings 99 | precision: 0.23 | recall: 0.31


Length of Test: 864
Number of real distress cases: 75


Final Model: Decision Tree or XGBoost?
I choose the XGBoost Classifier with COVID (all rows retained) as the final model due to comparable PR-AUC but higher recall.

## Final Model

In [35]:
train_all = model_df

final_model = xgb.XGBClassifier(max_depth = 3, n_estimators=200, learning_rate=0.05, random_state=500)
final_model.fit(train_all[features], train_all["target"])

# Latest month from the FULL table
latest = ipi_wide_model[ipi_wide_model["date"] == ipi_wide_model["date"].max()].dropna(subset=features).copy()
print(latest["date"].max(), len(latest)) # september 2026, 24 rows

# Predict and sort
latest["prob"] = final_model.predict_proba(latest[features])[:,1]
watch = latest.sort_values("prob", ascending=False)

watch["tier"] = np.where(watch["prob"] >= 0.13, "High",
                  np.where(watch["prob"] >= 0.10, "Watch", "Low"))

cols = ["division", "growth_yoy",  "yoy_roll_6", "distress", "prob", "tier"]
print(watch[cols].round(3))


2026-09-01 00:00:00 24
     division  growth_yoy  yoy_roll_6  distress   prob   tier
2960       30      -2.558       1.407       0.0  0.214   High
1550       20      -1.743       1.277       0.0  0.203   High
2819       29      10.551       4.204       0.0  0.176   High
1832       22       3.197       1.919       0.0  0.144   High
704        14       1.219       1.331       0.0  0.143   High
563        13       5.779      -0.083       0.0  0.135   High
845        15       1.216       3.015       0.0  0.119  Watch
986        16       3.701       2.782       0.0  0.087    Low
2255       25       4.087       3.793       0.0  0.069    Low
1127       17       4.724       4.083       0.0  0.067    Low
1268       18       6.534       6.702       0.0  0.053    Low
281        11       3.698       5.696       0.0  0.052    Low
1691       21       2.121       4.554       0.0  0.051    Low
140        10       7.691       3.145       0.0  0.051    Low
422        12       4.127       4.150       0.0

In [36]:
live = ipi_wide_model.dropna(subset=features).copy()
live = live[live["date"] >= "2026-04-01"]          # months with no target yet
live["prob"] = final_model.predict_proba(live[features])[:, 1]

hist = pooled_xgb[["date", "division", "target", "prob"]]
hist = pd.concat([hist.assign(source="backtest"),
                  live[["date", "division", "prob"]].assign(source="live")])

hist["tier"] = np.where(hist["prob"] >= 0.13, "High",
                np.where(hist["prob"] >= 0.10, "Watch", "Low"))
hist.to_parquet("dashboard/risk_history.parquet")
watch.to_parquet("dashboard/watchlist_latest.parquet")

# Features over time 
ipi_wide_model[["date", "division"] + features].to_parquet("dashboard/features.parquet")

# Sector names (standard manufacturing divisions 10 to 33)
names = {
"10":"Food products",
"11":"Beverages",
"12":"Tobacco products",
"13":"Textiles",
"14":"Wearing apparel",
"15":"Leather and related products",
"16":"Wood, products of wood and cork, except furniture",
"17":"Paper and paper products",
"18":"Printing and reproduction of recorded media",
"19":"Coke and refined petroleum products",
"20":"Chemicals and chemical products",
"21":"Pharmaceutical products and preparations",
"22":"Rubber and plastic products",
"23":"Non-metallic mineral products",
"24":"Basic metals",
"25":"Fabricated metal products, except machinery and equipment",
"26":"Computer, electronic and optical products",
"27":"Electrical equipment",
"28":"Machinery and equipment",
"29":"Motor vehicles, trailers and semi-trailers",
"30":"Other transport equipment",
"31":"Furniture",
"32":"Other manufacturing",
"33":"Repair and installation of machinery and equipment"}

pd.DataFrame({"division": list(names), "name": list(names.values())}) \
  .to_csv("dashboard/division_names.csv", index=False)